## Solving Cart Pole with Reinforce + Monte Carlo Policy Gradients

![Alt text](best_run.png)


In [ ]:
import os

import gymnasium as gym
import structlog
import torch
from reinforce_cart_pole_logger import ReinforceCartPoleLogger

from samsara_rl.control.policy_gradient.monte_carlo_policy_gradient import (
    MonteCarloPolicyGradient,
)
from samsara_rl.functions.neural_networks.fully_connected import (
    FullyConnected,
)
from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole

log = structlog.get_logger()

In [ ]:
log = structlog.get_logger()

MAX_EPISODES = 2800
EVAL_EPISODES = 20
BASE_DIR = "logs/reinforce_cart_pole_example"
ALPHAS = [0.0001, 0.0003, 0.001]  # 0.005, 0.001, 0.0005, 0.0001, 0.00005]
GAMMAS = [0.99]

In [ ]:
def run_experiment(env, alpha: float, gamma: float) -> None:
    """Train and evaluate a single configuration.

    Creates a LinearFunction and QLearningGradient agent, trains for
    MAX_EPISODES episodes, saves the learned weights, and runs greedy
    evaluation.

    Args:
        env: Gymnasium CartPole environment.
        alpha: Learning rate for the Q-learning update.
    """
    model_name = "reinforce"
    run_name = f"{model_name}_alpha={alpha}_gamma={gamma}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)

    fc = FullyConnected(4, 32, 2, dtype=torch.float32)

    agent = MonteCarloPolicyGradient(
        mdp=env, gamma=gamma, policy_network=fc, alpha=alpha, log_dir=run_dir, experiment_name=run_name
    )

    agent.register(ReinforceCartPoleLogger())

    agent.evaluate(max_iter=MAX_EPISODES)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [ ]:
def main() -> None:
    """Run grid search over feature configs, bias, and learning rates.

    Iterates over all combinations of FEATURE_CONFIGS, BIAS_CONFIGS,
    and ALPHAS, training and evaluating each one.
    """
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for alpha in ALPHAS:
        for gamma in GAMMAS:
            run_experiment(env, alpha, gamma)

In [ ]:
main()